![Header Image](../assets/header_image.png "Header Image")

# Devoir Optionnel 4 : Visualisation de Messages en Temps Réel avec ROS 2

Bienvenue dans ce tutoriel de visualisation de messages ROS 2 dans un Jupyter Notebook !
Cet exercice est facultatif. Nous le recommandons aux étudiants qui souhaitent approfondir
leur connaissance de la communication par messages en **ROS 2** et de la visualisation interactive.

> **Pré-requis :** Avoir complété le notebook `3_introduction_to_ros2_fr.ipynb`.
> Assurez-vous que le kernel **"Python 3.8 (ROS 2 Foxy)"** est sélectionné
> (**Kernel >> Change Kernel >> Python 3.8 (ROS 2 Foxy)**).

Dans ce devoir, vous allez

- **publier des messages ROS 2 en continu** dans un thread d'arrière-plan
- **visualiser des données en temps réel** avec `matplotlib` et `ipywidgets` dans Jupyter
- **comparer l'approche ROS 1 (JupyROS)** avec l'approche ROS 2 native (`rclpy`)
- **contrôler la publication** avec des widgets interactifs (curseurs d'amplitude et de fréquence)
- **publier différents types de signaux** : sinus, cosinus, onde carrée, et commandes de vitesse

# Approche ROS 1 vs ROS 2 pour la Visualisation

| Aspect | ROS 1 (JupyROS) | ROS 2 (rclpy natif) |
|--------|-----------------|---------------------|
| **Publication en boucle** | `%%thread_cell` (macro JupyROS) | `threading.Thread` standard Python |
| **Visualisation live** | `jupyros.live_plot()` | `matplotlib` + `ipywidgets.Output` |
| **Dépendance** | Bibliothèque JupyROS tierce | Python standard + rclpy |
| **Contrôle fin** | Limité par l'API JupyROS | Accès complet à matplotlib |
| **Nœud maître** | `roscore` requis | Aucun |

En ROS 2, nous n'avons pas besoin de `jupyros` : nous utilisons directement `rclpy` avec
les outils Python standard (`threading`, `matplotlib`, `ipywidgets`).

# Configurer l'environnement ROS 2

In [1]:
!source /opt/ros/foxy/setup.bash

# Configurer le chemin Python et vérifier le kernel

In [2]:
import sys
sys.path.insert(0, '/opt/ros/foxy/lib/python3.8/site-packages/')

import platform
print("Python utilisé :", platform.python_version())
assert platform.python_version_tuple()[1] == '8', \
    "ERREUR : mauvais kernel ! Sélectionnez 'Python 3.8 (ROS 2 Foxy)' dans Kernel >> Change Kernel."
print("Kernel OK — Python 3.8 confirmé.")

Python utilisé : 3.8.10
Kernel OK — Python 3.8 confirmé.


In [3]:
!pip install ipywidgets


# Importer les bibliothèques

In [ ]:
import subprocess, sys, importlib

# Installer ipywidgets pour Python 3.8 si absent
try:
    import ipywidgets
    print("ipywidgets déjà disponible.")
except ModuleNotFoundError:
    print("Installation de ipywidgets (première fois, ~10 s)...")
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'ipywidgets'])
    importlib.invalidate_caches()
    print("ipywidgets installé avec succès.")

import rclpy
from rclpy.node import Node
from geometry_msgs.msg import Vector3, Twist
from std_msgs.msg import Float64MultiArray

import threading
import time
import math
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

print("Bibliothèques importées avec succès !")

# Initialiser ROS 2 et créer le nœud

Contrairement à ROS 1, **aucun `roscore` n'est nécessaire**.
L'initialisation se fait entièrement en Python avec `rclpy.init()`.

In [ ]:
rclpy.init()
node = rclpy.create_node('visualisation_node')

spin_thread = threading.Thread(target=rclpy.spin, args=(node,), daemon=True)
spin_thread.start()

print(f"Nœud '{node.get_name()}' créé et thread ROS 2 démarré.")

# Publier des messages en continu avec un Thread

En ROS 1 avec JupyROS, la macro `%%thread_cell` lançait automatiquement la boucle de publication
dans un thread séparé. En ROS 2, nous utilisons directement **`threading.Thread`**.

Nous allons publier un message `Vector3` sur `/signal_stream` à une fréquence configurable :
- `x` : signal **sinus** (représente une composante oscillante)
- `y` : signal **cosinus** (en quadrature de phase avec x)
- `z` : signal **onde carrée** (utile pour les tests de latence)

Un `threading.Event` permet d'arrêter proprement le thread de publication.

In [ ]:
# Tampon de données partagé entre le thread de publication et la visualisation
data_lock = threading.Lock()
data_buffer = {'t': [], 'x': [], 'y': [], 'z': []}
MAX_POINTS = 200  # Nombre de points à conserver dans le tampon

# Paramètres de publication (modifiables via les curseurs plus bas)
pub_params = {'amplitude': 1.0, 'frequence': 1.0, 'actif': True}

# Événement pour arrêter les threads
stop_event = threading.Event()

# Créer le publisher Vector3
pub_signal = node.create_publisher(Vector3, '/signal_stream', 10)

def boucle_publication():
    """Publie des signaux sur /signal_stream en continu."""
    t0 = time.time()
    while not stop_event.is_set():
        t = time.time() - t0
        A = pub_params['amplitude']
        f = pub_params['frequence']
        omega = 2 * math.pi * f

        msg = Vector3()
        msg.x = A * math.sin(omega * t)          # Sinus
        msg.y = A * math.cos(omega * t)          # Cosinus
        msg.z = A * (1.0 if math.sin(omega * t) >= 0 else -1.0)  # Onde carrée

        pub_signal.publish(msg)

        with data_lock:
            data_buffer['t'].append(t)
            data_buffer['x'].append(msg.x)
            data_buffer['y'].append(msg.y)
            data_buffer['z'].append(msg.z)
            # Garder seulement les MAX_POINTS derniers points
            for k in data_buffer:
                if len(data_buffer[k]) > MAX_POINTS:
                    data_buffer[k] = data_buffer[k][-MAX_POINTS:]

        time.sleep(1.0 / 20.0)  # Publication à 20 Hz

pub_thread = threading.Thread(target=boucle_publication, daemon=True)
pub_thread.start()

print("Thread de publication démarré sur /signal_stream (20 Hz).")
print("Signaux publiés : x=sin, y=cos, z=onde carrée")

# Subscriber pour les messages Vector3

Nous créons un subscriber qui reçoit les messages et affiche des statistiques.
Ceci remplace `jupyros.subscribe()` de JupyROS.

In [ ]:
compteur_messages = [0]

def callback_signal(msg):
    compteur_messages[0] += 1
    if compteur_messages[0] % 20 == 0:  # Afficher 1 message sur 20
        print(f"[#{compteur_messages[0]:4d}] x={msg.x:+.3f}  y={msg.y:+.3f}  z={msg.z:+.3f}")

sub_signal = node.create_subscription(Vector3, '/signal_stream', callback_signal, 10)

print("Subscriber actif sur /signal_stream.")
print("Attendez 2 secondes pour voir les messages arriver...")
time.sleep(2.0)
print(f"Messages reçus : {compteur_messages[0]}")

# Visualisation en Temps Réel avec matplotlib + ipywidgets
#
# Le thread de publication tourne en arrière-plan et remplit data_buffer.
# La cellule suivante lit ce tampon et affiche un graphique statique.
# Ré-exécutez la cellule de visualisation autant de fois que vous voulez
# pour voir l'évolution des signaux — c'est plus fiable que matplotlib
# dans un thread (matplotlib n'est pas thread-safe).

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

# Attendre que le buffer soit rempli
print("Collecte de données pendant 3 secondes...")
time.sleep(3.0)

# Lire le tampon depuis le thread principal (thread-safe avec le lock)
with data_lock:
    t_snap  = list(data_buffer['t'])
    x_snap  = list(data_buffer['x'])
    y_snap  = list(data_buffer['y'])
    z_snap  = list(data_buffer['z'])

print(f"{len(t_snap)} points collectés — affichage du snapshot...")

fig, axes = plt.subplots(3, 1, figsize=(11, 6), sharex=True)
fig.suptitle('Signaux ROS 2 — /signal_stream (snapshot)', fontsize=13)

axes[0].plot(t_snap, x_snap, color='steelblue',  linewidth=1.5, label='sin (x)')
axes[0].set_ylabel('sin (x)'); axes[0].set_ylim(-1.6, 1.6)
axes[0].axhline(0, color='gray', linewidth=0.5); axes[0].grid(True, alpha=0.3)

axes[1].plot(t_snap, y_snap, color='darkorange', linewidth=1.5, label='cos (y)')
axes[1].set_ylabel('cos (y)'); axes[1].set_ylim(-1.6, 1.6)
axes[1].axhline(0, color='gray', linewidth=0.5); axes[1].grid(True, alpha=0.3)

axes[2].plot(t_snap, z_snap, color='green',      linewidth=1.5, label='carré (z)')
axes[2].set_ylabel('carré (z)'); axes[2].set_ylim(-1.6, 1.6)
axes[2].set_xlabel('Temps (s)')
axes[2].axhline(0, color='gray', linewidth=0.5); axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
print("Ré-exécutez cette cellule pour voir un nouveau snapshot.")

# Contrôles Interactifs : Amplitude et Fréquence
#
# Ces curseurs modifient les paramètres du signal en temps réel.
# Après avoir bougé un curseur, ré-exécutez la cellule de snapshot
# ci-dessus pour voir l'effet sur les signaux publiés.

In [ ]:
slider_amplitude = widgets.FloatSlider(
    value=1.0, min=0.1, max=3.0, step=0.1,
    description='Amplitude :',
    style={'description_width': '100px'},
    layout=widgets.Layout(width='450px')
)
slider_frequence = widgets.FloatSlider(
    value=1.0, min=0.1, max=5.0, step=0.1,
    description='Fréquence (Hz) :',
    style={'description_width': '100px'},
    layout=widgets.Layout(width='450px')
)

def maj_params(change):
    pub_params['amplitude'] = slider_amplitude.value
    pub_params['frequence'] = slider_frequence.value
    with data_lock:
        for k in data_buffer:
            data_buffer[k].clear()  # Effacer l'historique lors d'un changement

slider_amplitude.observe(maj_params, names='value')
slider_frequence.observe(maj_params, names='value')

display(widgets.VBox([
    widgets.Label('Modifiez ces paramètres pour changer les signaux en temps réel :'),
    slider_amplitude,
    slider_frequence
]))

print("Contrôles affichés. Déplacez les curseurs pour changer les signaux.")

# Arrêter la Publication
#
# Exécutez cette cellule pour stopper le thread de publication des signaux.
# Les threads de commandes et de visualisation restent actifs
# jusqu'à la cellule d'arrêt final en bas du notebook.

In [ ]:
stop_event.set()
time.sleep(0.3)
print("Thread de publication des signaux arrêté.")

# Cas d'Usage : Commandes de Vitesse d'un Robot Mobile (Twist)

Le message `Twist` est le type le plus utilisé pour contrôler les robots mobiles en ROS 2.
Il combine :
- **Vitesse linéaire** (`linear.x`) : avance/recul en m/s
- **Vitesse angulaire** (`angular.z`) : rotation en rad/s

Nous simulons ici les commandes envoyées par un contrôleur PID à un robot différentiel,
suivant une trajectoire en forme de spirale.

In [ ]:
pub_cmd = node.create_publisher(Twist, '/cmd_vel', 10)

# Tampon pour les commandes de vitesse
cmd_buffer = {'t': [], 'linear_x': [], 'angular_z': [], 'x_pos': [0.0], 'y_pos': [0.0]}

stop_cmd = threading.Event()

def boucle_commandes():
    t0 = time.time()
    x, y, theta = 0.0, 0.0, 0.0
    dt = 0.05
    while not stop_cmd.is_set():
        t = time.time() - t0

        # Trajectoire spirale : vitesse linéaire constante, angulaire diminue
        v_lin = 0.3  # m/s
        v_ang = 1.5 / (1.0 + 0.3 * t)  # rad/s, diminue avec le temps

        msg = Twist()
        msg.linear.x  = v_lin
        msg.angular.z = v_ang
        pub_cmd.publish(msg)

        # Intégration de la position (odométrie simplifiée)
        theta += v_ang * dt
        x     += v_lin * math.cos(theta) * dt
        y     += v_lin * math.sin(theta) * dt

        cmd_buffer['t'].append(t)
        cmd_buffer['linear_x'].append(v_lin)
        cmd_buffer['angular_z'].append(v_ang)
        cmd_buffer['x_pos'].append(x)
        cmd_buffer['y_pos'].append(y)

        time.sleep(dt)
        if t > 15.0:  # Arrêt automatique après 15 secondes
            break

cmd_thread = threading.Thread(target=boucle_commandes, daemon=True)
cmd_thread.start()

print("Publication de commandes Twist sur /cmd_vel (15 secondes)...")
cmd_thread.join()
print(f"Simulation terminée. {len(cmd_buffer['t'])} commandes publiées.")

# Visualiser la Trajectoire du Robot

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Graphique gauche : vitesses en fonction du temps
axes[0].plot(cmd_buffer['t'], cmd_buffer['linear_x'],  label='v_lin (m/s)',  color='steelblue')
axes[0].plot(cmd_buffer['t'], cmd_buffer['angular_z'], label='v_ang (rad/s)', color='darkorange')
axes[0].set_title('Commandes Twist — /cmd_vel')
axes[0].set_xlabel('Temps (s)')
axes[0].set_ylabel('Vitesse')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Graphique droit : trajectoire X/Y
x_traj = cmd_buffer['x_pos'][1:]
y_traj = cmd_buffer['y_pos'][1:]
sc = axes[1].scatter(x_traj, y_traj,
                     c=cmd_buffer['t'], cmap='plasma',
                     s=5, linewidths=0)
axes[1].set_title('Trajectoire du robot (odométrie intégrée)')
axes[1].set_xlabel('x (m)')
axes[1].set_ylabel('y (m)')
axes[1].set_aspect('equal')
axes[1].grid(True, alpha=0.3)
plt.colorbar(sc, ax=axes[1], label='Temps (s)')

plt.tight_layout()
plt.show()
print("Trajectoire spirale affichée — couleur = progression temporelle.")

# Cas d'Usage : Publication Multi-Canal avec Float64MultiArray

Le message `Float64MultiArray` est utile pour transmettre des tableaux de données arbitraires,
par exemple les mesures simultanées de plusieurs capteurs (IMU 6-axes, tensions de batteries, etc.).

Nous publions ici les valeurs de 4 signaux (sinus à différentes phases) comme si
c'étaient 4 capteurs de distance embarqués sur un véhicule.

In [ ]:
pub_multi = node.create_publisher(Float64MultiArray, '/capteurs_distance', 10)

N_CAPTEURS = 4
phases = [0.0, math.pi/4, math.pi/2, 3*math.pi/4]  # Déphasages des capteurs
noms_capteurs = ['Avant-Gauche', 'Avant-Droit', 'Arrière-Gauche', 'Arrière-Droit']

# Collecter 3 secondes de données
historique_multi = [[] for _ in range(N_CAPTEURS)]
temps_multi = []

t0 = time.time()
while time.time() - t0 < 3.0:
    t = time.time() - t0
    distances = [2.0 + math.sin(2 * math.pi * 0.5 * t + phi) for phi in phases]

    msg = Float64MultiArray()
    msg.data = distances
    pub_multi.publish(msg)

    temps_multi.append(t)
    for i, d in enumerate(distances):
        historique_multi[i].append(d)

    time.sleep(0.05)  # 20 Hz

print(f"{len(temps_multi)} échantillons publiés sur /capteurs_distance")

# Visualisation
fig, ax = plt.subplots(figsize=(11, 4))
couleurs = ['steelblue', 'darkorange', 'green', 'red']
for i in range(N_CAPTEURS):
    ax.plot(temps_multi, historique_multi[i],
            label=noms_capteurs[i], color=couleurs[i], linewidth=1.5)
ax.set_title('Float64MultiArray — /capteurs_distance (4 capteurs de distance)')
ax.set_xlabel('Temps (s)')
ax.set_ylabel('Distance (m)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Vérifier les Topics depuis le Terminal

Pendant que les publishers sont actifs, ouvrez un terminal JupyterLab
(**Fichier >> Nouveau >> Terminal**) et exécutez :

```bash
source /opt/ros/foxy/setup.bash

# Lister tous les topics actifs
ros2 topic list

# Voir les messages en temps réel
ros2 topic echo /signal_stream

# Afficher les statistiques de publication (fréquence, bande passante)
ros2 topic hz /signal_stream
ros2 topic bw /signal_stream

# Voir les informations sur un topic
ros2 topic info /cmd_vel
```

> **Rappel :** Ces commandes fonctionnent sans roscore, grâce à DDS.

# Arrêter tous les Threads et libérer les Ressources

In [ ]:
stop_event.set()
stop_cmd.set()
time.sleep(0.3)

node.destroy_node()
rclpy.shutdown()

print("Tous les threads arrêtés.")
print("Nœud ROS 2 détruit — ressources DDS libérées.")

# Résumé

- Vous avez remplacé **`%%thread_cell`** (JupyROS) par `threading.Thread` standard Python pour publier des messages ROS 2 en continu.
- Vous avez remplacé **`jupyros.live_plot()`** par une visualisation matplotlib dans un `ipywidgets.Output`, offrant un contrôle total sur l'affichage.
- Vous avez ajouté des **contrôles interactifs** (`ipywidgets.FloatSlider`) pour modifier amplitude et fréquence en temps réel.
- Vous avez publié des **commandes de vitesse** (`Twist`) et visualisé la trajectoire résultante.
- Vous avez utilisé **`Float64MultiArray`** pour transmettre des données multi-capteurs.
- Toute cette fonctionnalité est disponible **sans `roscore`** et sans bibliothèque tierce comme JupyROS.